# Fáze 1 — Plný vzorek

Podle `MasterOutline.md` §4 (Fáze 1) a §5 (otevřená rozhodnutí). `network_type="drive"` je finální volba napříč projektem (viz `00_pilot.ipynb`, sekce srovnání `drive` vs. `all`, a `MasterOutline.md` §5) — `"all"` na velkém městě (Brno) opakovaně selhávalo na veřejném Overpass serveru i po zvýšení timeoutu a retry logice, takže riziko nespolehlivého stahování převážilo nad hustší sítí u malých sídel.

Pipeline je rozdělený na dvě fáze, ať se síťové riziko nemíchá s výpočtem: **"Stáhni vše"** (jediná buňka, co se dotýká sítě) a pak **"Spočítej metriky"** (čistě offline, bezpečné spouštět opakovaně). Grafy a metriky se ukládají lokálně (`data/raw/graphs/`, `data/processed/sample_metrics.csv`), takže druhý běh už nic nestahuje ani nepočítá znovu.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import DATA_PROCESSED
from src.viz import set_style, save_fig
from src.streets.osm import download_graph, download_all
from src.streets.metrics import compute_metrics
from src.streets.clustering import build_feature_matrix, sweep_k, fit_kmeans, project_2d
from src.streets.viz import plot_polar_grid, plot_silhouette_scores, plot_cluster_scatter

set_style()

## Vzorek

Celý `sample_seed.csv` (33 sídel) — viz `MasterOutline.md` §3.1 pro složení vzorku.

In [ ]:
sample = pd.read_csv(DATA_PROCESSED / "sample_seed.csv", encoding="utf-8")
len(sample), sample[["name", "district_name", "region_name", "population_total"]]

## Stáhni vše (jediná síťová část celého notebooku)

Tahle buňka jen stahuje/cachuje `.graphml` grafy — nic nepočítá. Je to záměrně oddělené od výpočtu metrik: je to jediné místo v celém pipeline, co se vůbec dotýká sítě, takže je to jediné místo, kde může nastat problém s Overpassem (viz `MasterOutline.md` §5 — známé, ne úplně vyřešené potíže s tímhle konkrétním připojením). Spusť ji sám, až budeš mít čas počkat — u velkého vzorku to může trvat.

Jedno selhavší město nezastaví zbytek dávky (zaloguje se a pokračuje dál) — funkce vrátí seznam těch, co se nepovedly, ty pak stačí spustit znovu (zbytek už je stažený, nestahuje se znovu).

Grafy se ukládají do `data/raw/graphs/` — jakmile tahle buňka doběhne (i s pár neúspěchy), všechno ostatní v notebooku je čistě offline výpočet bez síťového rizika.

In [ ]:
network_types = ["drive"]

failed = download_all(sample, network_types=network_types)
print(f"\n{len(failed)} selhalo: {failed}" if failed else "\nvšechno staženo")

## Spočítej metriky (čistě offline)

Od tohohle místa dál se pipeline vůbec nedotýká sítě — čte jen grafy z `data/raw/graphs/`, co stáhla buňka výše. Bezpečné spouštět opakovaně (třeba po opravě chyby v `metrics.py`) bez rizika, že narazíš na Overpass znovu — díky cache v `sample_metrics.csv` se navíc nic nepřepočítává vícekrát.

In [ ]:
metrics_path = DATA_PROCESSED / "sample_metrics.csv"

all_metrics = compute_metrics(sample, network_types=network_types, csv_path=metrics_path)
all_metrics

## Shlukování — volba k

Feature matice ze standardizovaných metrik (viz `MasterOutline.md` §3.3). Silhouette score pro rozsah `k` napoví, kolik shluků dává smysl — nejde jen mechanicky vzít maximum, mrkni i na velikost/interpretovatelnost jednotlivých shluků.

In [ ]:
X, clean = build_feature_matrix(all_metrics)

k_scores = sweep_k(X, range(2, 9))
plot_silhouette_scores(k_scores)

## Finální shlukování

Nastav `k` podle grafu výše, než pojedeš dál.

In [ ]:
k = 4  # <- rozhodni podle silhouette plotu výše

model, score = fit_kmeans(X, k)
clean = clean.copy()
clean["cluster"] = model.labels_
print(f"k={k}, silhouette={score:.3f}")
clean[["name", "network_type", "cluster"] + list(X.columns)]

In [ ]:
coords = project_2d(X)
plot_cluster_scatter(coords, clean["cluster"], names=clean["name"])

## Poster — finální vizualizace

Polar grid pro celý vzorek, uspořádaný podle shluku — hlavní výstupní obrázek pro web/článek (viz `MasterOutline.md` §3.4, §8).

In [ ]:
ordered = clean.sort_values("cluster")
graphs = {
    f"{row['name']} (c{row['cluster']})": download_graph(sample[sample['name'] == row['name']].iloc[0], network_type="drive")
    for _, row in ordered.iterrows()
}

fig = plot_polar_grid(graphs, num_bins=36, ncols=6, bg="white")
save_fig(fig, "01_full_sample_poster")

## Další kroky

- Interpretuj shluky (`clean.groupby("cluster")[...].mean()`) vzhledem k RQ2/RQ3 z `MasterOutline.md` §2 — odpovídají kraji, velikosti, historickému typu?
- Pokud dává smysl vzorek rozšířit (Fáze 1 cíl je ~50–100 sídel, teď máme 33), přidej řádky do `sample_seed.csv` a spusť `download_all()` + `compute_metrics()` znovu — nic z hotového se nepřepočítá.
- Binning bearingu (`num_bins`) je pořád otevřené rozhodnutí (`MasterOutline.md` §5) — zkus 16/32 a porovnej čitelnost posteru.